# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya — Exploration with `mlcroissant`  
This notebook demonstrates how to explore the FAIR² dataset documenting ordered logistic regression results for factors influencing the adoption of indigenous and modern knowledge in rangeland management in Northern Kenya. We'll use the [`mlcroissant`](https://mlcommons.github.io/croissant) Python library to interact with the dataset using its Croissant schema.

### Dataset Source
The dataset metadata is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata using `mlcroissant` and display its description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display dataset name and description from metadata
print(f"Dataset Title: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview
List the available record sets, fields, and their `@id`s. 
We'll programmatically display all record sets, and for each, their fields (columns).

In [ ]:
# List all record sets by @id and name
print("Available Record Sets:")
for rs in dataset.record_sets:
    info = f"- @id: {rs['@id']} | name: {rs.get('name', '<no name>')}"
    print(info)

# For each record set, list fields (columns) by @id and name
for rs in dataset.record_sets:
    print(f"\nRecordSet: {rs.get('name', '<no name>')} (@id: {rs['@id']})")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        for field in fields:
            # field may be dict or str, resolve if str (should be @id-ref), else show @id directly
            if isinstance(field, dict):
                print(f"  - Field @id: {field.get('@id', str(field))} | name: {field.get('name', '<no name>')}")
            else:
                print(f"  - Field @id: {field}")
    else:
        print("  (No fields listed)")

## 3. Data Extraction
Select one or multiple record sets and load their data into Pandas DataFrames for exploration.

Make sure to use the `@id` of the record set and fields/columns (from the previous step).

In [ ]:
# List all record set @ids and select those to extract
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = dict()

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Columns (@ids): {list(df.columns)}")
    else:
        print(f"No records found for {record_set_id}.")

# For demonstration, pick the first record set with records (replace @id below if known)
main_rs_id = next(iter(dataframes.keys())) if dataframes else None

if main_rs_id:
    print(f"\nPreview of RecordSet {main_rs_id}:")
    display(dataframes[main_rs_id].head())
else:
    print("No record sets with data to extract.")

## 4. Exploratory Data Analysis (EDA)
Apply typical data cleaning and processing: 
- Filter numeric values
- Normalize numeric fields
- Optionally, group by a categorical field

Demonstration below uses the first available numeric field in the selected record set.

In [ ]:
if main_rs_id:
    df = dataframes[main_rs_id]

    # Try to auto-detect first numeric column for demo
    numeric_fields = df.select_dtypes(include=['number', 'float', 'int']).columns.tolist()
    if not numeric_fields and len(df) > 0:
        # attempt conversion in case columns stored as str
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except:
                pass
        numeric_fields = df.select_dtypes(include=['number', 'float', 'int']).columns.tolist()
    
    if numeric_fields:
        numeric_field = numeric_fields[0]
        threshold = df[numeric_field].mean() if df[numeric_field].notna().any() else 1
        # If the column has all NaN/None, skip
        filtered_df = df[df[numeric_field] > threshold] if not df[numeric_field].isnull().all() else df
        print(f"Filtered records where {numeric_field} > {threshold:.3f} (using field @id):")
        display(filtered_df.head())
        # Normalize
        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized values for {numeric_field}:")
        display(filtered_df[[numeric_field, normalized_col]].head())
        # Try grouping by first non-numeric field if available
        group_fields = df.select_dtypes(exclude=['number', 'float', 'int']).columns.tolist()
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No categorical fields available for grouping.")
    else:
        print("No numeric fields identified for EDA in this record set.")
else:
    print("No main record set DataFrame available.")

## 5. Visualization
Visualize distributions or relationships of variables using basic plots. We'll use the first numeric field and, if available, group by a categorical variable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_fields:
    plt.figure(figsize=(7,3))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field} (@id)")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
    # If grouped by category possible, show barplot
    if group_fields:
        plt.figure(figsize=(8,4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field} (@id)")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion

In this notebook, we have:
- Loaded and explored the metadata of the FAIR² rangeland management adoption dataset via its Croissant schema
- Enumerated record sets and their available fields by their `@id`
- Loaded tabular data for analysis and demonstrated basic filtering, normalization, and grouping operations
- Visualized distributions and group comparisons using Matplotlib and Seaborn

**Next Steps:** 
- Review the record sets for further analysis relevant to your use case
- Consult the original dataset schema and documentation for variable definitions and recommended usages
- Apply additional pre-processing, feature engineering, or modeling as required for your research

For more, visit the [mlcroissant documentation](https://mlcommons.github.io/croissant).